# 1. Data Loading and Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
df = pd.read_csv("datasets/House_Price.csv", header = 0)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 1.1. Univariate Analysis
### 1.1.1. Extended Data Dictionary (EDD)

In [ ]:
df.describe()

In [ ]:
sns.jointplot(x = "n_hot_rooms", y = "price", data = df)

In [ ]:
sns.jointplot(x = "rainfall", y = "price", data = df)

In [ ]:
df.head()

In [ ]:
# sns.countplot is used for categorical variables.
sns.countplot(x = "airport", data = df)

In [ ]:
sns.countplot(x = "waterbody", data = df)

In [ ]:
sns.countplot(x = "bus_ter", data = df)

#### **Observations from EDD:**
1. Missing values in `n_hos_beds`.
2. Skewness or outliers in `crime_rate`.
3. Outliers in `n_hot_rooms` and `rainfall`.
4. `bus_ter` is taking only `YES` values.

### 1.1.2. Outlier Treatment

In [ ]:
df.info()

**Using Capping & Flooring:**
* Impute all the values above 3* P99 and below 0.3*P1
* Impute with values 3* P99 and 0.3*P1
* You can use any multiplier instead of 3, as per your business requirement

In [ ]:
# For Upper Value (uv)
# The upper value case can only be seen in n_hot_rooms.

np.percentile(df.n_hot_rooms, [99])

In [ ]:
np.percentile(df.n_hot_rooms, [99])[0]

In [ ]:
uv = np.percentile(df.n_hot_rooms, [99])[0]

In [ ]:
# Checking all the values greater than the upper value (uv)
df[df.n_hot_rooms > uv]

In [ ]:
# Capping the values
# Taking multiplication value (n) = 3 since we only want to treat 101.12 and 81.12 outliers and keep the remaining same as they are near to the uv value.

# df.n_hot_rooms[(df.n_hot_rooms > 3 * uv)] = 3 * uv # Old usage to assign the value to the df.

df.loc[df.n_hot_rooms > 3 * uv, "n_hot_rooms"] = 3 * uv

In [ ]:
df[df.n_hot_rooms > uv]

In [ ]:
# For Lower Value (lv)
# The lower value case can only be seen in rainfall.

np.percentile(df.rainfall, [1])[0]

In [ ]:
lv = np.percentile(df.rainfall, [1])[0]

In [ ]:
df[(df.rainfall < lv)]

In [ ]:
# df.rainfall[(df.rainfall < 0.3 * lv)] = 0.3 * lv
df.loc[df.rainfall < 0.3 * lv, "rainfall"] = 0.3 * lv

In [ ]:
df[(df.rainfall < lv)]

In [ ]:
# Analyze the crime_rate
sns.jointplot(x = "crime_rate", y = "price", data = df)

**Observations:**
1. The relationship between `crime_rate` and `price` is not linear.
2. Looks like a polynomial relationship.
3. To make it more linear, we can use one of the following ways:
	* Taking Log.
    * Taking Exponential.
    * Square root.
4. After this the outliers will be automatically gone.
5. Variable Transformation is needed when there is no linearity.

In [ ]:
df.describe()

### 1.1.3. Mising Value Imputation

* To check all the missing values, `df.info()` is used instead of EDD using `df.describe()` because EDD does not tell about the categorical data, i.e. `waterbody` in this case.

In [ ]:
df.info()

In [ ]:
df["waterbody"] = df["waterbody"].fillna("No Waterbody")
# This is used because in missing value imputation when we checked that in CSV, "None" is used in cases where there is no water-body and this "None" is treated as missing value.

In [ ]:
df.info()

In [ ]:
df.n_hos_beds = df.n_hos_beds.fillna(df.n_hos_beds.mean())  # Filling missing values with mean

In [ ]:
df.info()

## 1.2. Bivariate Analysis
### 1.2.1. Variable Transformation & Deletion

In [ ]:
sns.jointplot(x = "crime_rate", y = "price", data = df)

**Observations:**
* The curve looks like a logarithmic curve.
* Transform this curve to have a linear relationship between `crime_rate` & `price`.
* Take log of `x` variable i.e. `crime_rate` so that we can have a linear relationship.
* Most of the values are near 0 and `log 0` is not defined, and it tends towards minus infinity, so we will add 1 as a constant in our variable.

In [ ]:
df.crime_rate = np.log(1 + df.crime_rate)

In [ ]:
sns.jointplot(x = "crime_rate", y = "price", data = df)

* `dist1`, `dist2`, `dist3` and `dist4` conveys the same information that is the distance from the employment hub.
* Transform them to a single variable to show the average of the 4 distances.
* New variables are created as per the business requirements. For example if people are looking for nearest employment hub, we can use the lowest dist value using:

```python
df["min_dist"] = df[["dist1", "dist2", "dist3", "dist4"]].min(axis=1)
```
where,
* `axis=1` means calculate across columns for each row.
* It picks the minimum value among the 4 distance columns.

In [ ]:
df["avg_dist"] = (df.dist1 + df.dist2 + df.dist3 + df.dist4) / 4

In [ ]:
df.describe()

In [ ]:
del df["dist1"]
del df["dist2"]
del df["dist3"]
del df["dist4"]

In [ ]:
df.describe()

In [ ]:
# Deleting bus terminal (bus_ter) variable as well since it uses only one value (YES) and does not give any relevant information.
del df["bus_ter"]

In [ ]:
df.head()

## 1.3. Dummy Variable

In [ ]:
df = pd.get_dummies(df, dtype = int)
# Used dtype = int here as it creates bool value by default.

In [ ]:
df.head()

**Observations:**
* As it is clear from the data that the `airport_NO` and `airport_YES` have full negative correlation as `airport_YES = 1` shows presence of the airport and vice versa, so we can delete the `airport_NO` column.
* Delete other such columns too.

In [ ]:
del df["airport_NO"]
del df["waterbody_No Waterbody"]

In [ ]:
df.head()

## 1.4. Correlation Matrix

In [ ]:
df.corr()

**Observations:**
* `room_num` has a very high correlation with `price`.
* `poor_prop` has high negative correlation.
* Insignificant variables can be deleted if there are large number of variables (100+).
* High correlation between 2 independent variables leads to the problem of **multiple collinearity**.
* Find out all the variables with high correlation i.e. `values > 0.8` or `value < -0.8`.
	* `parks` vs `air_qual`
* Delete one of the variables by:
	* Check correlation of both highly correlated independent with the dependent `y` variable i.e. `price` in this case.
    * Delete the variable having low correlation with independent variable i.e. delete `park` in this case.

In [ ]:
del df["parks"]

In [ ]:
df.head()

# 2. Simple Linear Regression
## 2.1. Using `statsmodel`
* For this case, we are using `room_num` as independent variable `X` and `price` as dependent variable `y`.

In [ ]:
from statsmodels import api as sn

* Adding a constant in dependent variable
* `statsmodel` does not use a constant term by default.
* It means that the intercept (Beta0) will be 0 by default.
* We are creating the intercept variable (Beta0)

In [ ]:
X = sn.add_constant(df["room_num"])

In [ ]:
# Fitting the model
# lm = model object
# OLS = Ordinary Least Square
lm = sn.OLS(df["price"], X).fit()

In [ ]:
lm.summary()

**Complete OLS Regression Results Explanation**

This output is from an **OLS (Ordinary Least Squares) Linear Regression** model using the `statsmodels` library.

The model predicts:

- **Dependent Variable (Target):** `price`
- **Independent Variable (Feature):** `room_num`

---

**Regression Equation**

General regression equation:

$$
Y = \beta_0 + \beta_1X
$$

For this model:

$$
price = -34.6592 + 9.0997(room\_num)
$$

Where:

- $\beta_0$ = intercept (constant)
- $\beta_1$ = slope coefficient

---

1. **Dependent Variable**

**Dep. Variable: `price`**

This is the target variable the model tries to predict.

The model learns how `room_num` affects house price.

---

2. **Model Type**

**Model: `OLS`**

OLS = Ordinary Least Squares.

OLS finds the best-fit regression line by minimizing the squared residual errors.

Residual/Error:

$$
Residual = Actual - Predicted
$$

OLS minimizes:

$$
\sum (y_i - \hat{y}_i)^2
$$

Where:

- $y_i$ = actual value
- $\hat{y}_i$ = predicted value

Errors are squared because:

- negative errors become positive
- large errors are penalized more strongly
- easier mathematical optimization

---

3. **Method**

**Method: Least Squares**

The regression line is selected such that the total squared error becomes minimum.

---

4. **Number of Observations**

**No. Observations: `506`**

The dataset contains 506 rows.

So the model was trained using 506 samples.

---

5. **Degrees of Freedom**

---

**Df Model: `1`**

This means there is 1 predictor variable:

```python
room_num
```

---

**Df Residuals: `504`**

Formula:

$$
Df_{Residual} = n - p - 1
$$

Where:

- $n$ = number of observations
- $p$ = number of predictors

Calculation:

$$
506 - 1 - 1 = 504
$$

Residual degrees of freedom represent remaining independent information after fitting the model.

---

6. **R-squared**

**R-squared = `0.485`**

R² measures how much variation in the target variable is explained by the model.

Formula:

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

Where:

- $SS_{res}$ = residual sum of squares
- $SS_{tot}$ = total sum of squares

Interpretation:

- 0 → model explains nothing
- 1 → perfect prediction

Here:

$$
R^2 = 0.485
$$

Meaning:

- 48.5% of variation in house prices is explained by `room_num`
- 51.5% variation remains unexplained

This is decent for a single-variable regression.

---

7. **Adjusted R-squared**

**Adj. R-squared = `0.484`**

Adjusted R² penalizes unnecessary variables.

Formula:

$$
Adjusted\ R^2 = 1-(1-R^2)\frac{n-1}{n-p-1}
$$

Normal R² always increases when new variables are added, even useless ones.

Adjusted R² increases only when predictors genuinely improve the model.

Since there is only one predictor:

- R² = 0.485
- Adjusted R² = 0.484

Both are almost identical.

---

8. **F-statistic**

**F-statistic = `474.3`**

Tests whether the overall regression model is statistically useful.

**Hypotheses:**

- **Null Hypothesis ($H_0$)**

$$
\beta_1 = 0
$$

The predictor has no effect.

- **Alternative Hypothesis ($H_1$)**

$$
\beta_1 \neq 0
$$

The predictor affects the target.

Large F-statistic means:

- the model explains significant variation
- regression relationship is strong

  474.3 is extremely high.

---

9. **Prob(F-statistic)**

**Prob(F-statistic) = `1.31e-74`**

This is the p-value for the overall regression model.

Very tiny value:

$$
1.31 \times 10^{-74}
$$

Interpretation:

- model is highly significant
- reject null hypothesis
- `room_num` significantly affects `price`

Rule:

- p-value < 0.05 → statistically significant

---

10. **Coefficient Table**

| Variable | coef     | std err | t       | P>\|t\| |
| -------- | -------- | ------- | ------- | ------- |
| const    | -34.6592 | 2.642   | -13.118 | 0.000   |
| room_num | 9.0997   | 0.418   | 21.779  | 0.000   |

---

11. **Intercept (const)**

**coef = `-34.6592`**

This is the intercept.

Meaning:
When `room_num = 0`:

$$
price = -34.6592
$$

Practically unrealistic because houses cannot have zero rooms.

Intercept mainly helps position the regression line mathematically.

---

12. **Standard Error of Intercept**

**std err = `2.642`**

This measures uncertainty in the intercept estimate.

Interpretation:

- smaller standard error → more reliable estimate
- larger standard error → less certainty

The intercept estimate may vary approximately by ±2.642 units.

---

13. **t-statistic for Intercept**

**t = `-13.118`**

Formula:

$$
t = \frac{Coefficient}{Standard\ Error}
$$

Calculation:

$$
t = \frac{-34.6592}{2.642} \approx -13.118
$$

Large absolute t-value means the intercept is statistically different from zero.

---

14. **P-value for Intercept**

**P>|t| = `0.000`**

Very small p-value indicates:

- intercept is statistically significant
- intercept is unlikely to be zero

---

15. \*\*Confidence Interval for Intercept

**95% CI = `[-39.850, -29.468]`**

Meaning:
With 95% confidence, the true intercept lies between:

- -39.850
- -29.468

Since 0 is not inside the interval:

- intercept is statistically significant

---

16. **Coefficient of `room_num`**

**coef = `9.0997`**

This is the slope coefficient.

Meaning:
For every increase of 1 room:

$$
price \ increases \ by \ approximately \ 9.1 \ units
$$

This is the main learning from the model.

---

17. **Standard Error of `room_num`**

**std err = `0.418`**

Measures uncertainty in the slope estimate.

Smaller standard error means:

- coefficient estimate is stable
- model is confident about the relationship

Since 0.418 is relatively small compared to 9.0997:

- the coefficient estimate is reliable

---

18. **t-statistic for `room_num`**

**t = `21.779`**

Formula:

$$
t = \frac{Coefficient}{Standard\ Error}
$$

Calculation:

$$
t = \frac{9.0997}{0.418} \approx 21.779
$$

Very large t-value means:

- `room_num` strongly contributes to prediction
- relationship is statistically significant

---

19. **P-value for `room_num`**

**P>|t| = `0.000`**

Hypotheses:

- **Null Hypothesis**

$$
\beta_1 = 0
$$

- **Alternative Hypothesis**

$$
\beta_1 \neq 0
$$

Very small p-value means:

- reject null hypothesis
- `room_num` significantly affects house price

---

20. **Confidence Interval for `room_num`**

**95% CI = `[8.279, 9.921]`**

Meaning:
With 95% confidence, the true slope lies between:

- 8.279
- 9.921

Since 0 is not inside the interval:

- the predictor is statistically significant

---

21. **Log-Likelihood**

**Log-Likelihood = `-1671.6`**

Measures how likely observed data is under the model.

Higher values (less negative) are better.

Mostly useful for comparing models.

---

22. **AIC**

**AIC = `3347`**

AIC = Akaike Information Criterion.

Used for model comparison.

Formula includes:

- goodness of fit
- penalty for complexity

Lower AIC is better.

---

23. **BIC**

**BIC = `3356`**

BIC = Bayesian Information Criterion.

Similar to AIC but penalizes model complexity more strongly.

Lower BIC is better.

---

24. **Omnibus Test**

**Omnibus = `103.753`**

Tests whether residuals are normally distributed.

Large value suggests:

- residuals are not normal

---

25. **Prob(Omnibus)**

**Prob(Omnibus) = `0.000`**

Very small p-value means:

- residuals are not normally distributed
- regression assumption is violated

---

26. **Jarque-Bera Test**

**Jarque-Bera (JB) = `633.429`**

Another normality test.

High value indicates:

- skewness
- heavy tails
- outliers

Residuals are not normally distributed.

---

27. **Prob(JB)**

**Prob(JB) = `2.84e-138`**

Extremely small p-value confirms:

- residuals are not normal

---

28. **Skew**

**Skew = `0.729`**

Measures asymmetry of residual distribution.

Interpretation:

- 0 → symmetric
- positive → right-skewed
- negative → left-skewed

Residuals are positively skewed.

---

29. **Kurtosis**

**Kurtosis = `8.284`**

Measures heaviness of tails.

Normal distribution kurtosis ≈ 3.

Since:

$$
8.284 > 3
$$

Residuals contain:

- heavy tails
- potential outliers

---

30. **Durbin-Watson Statistic**

**Durbin-Watson = `0.681`**

Tests autocorrelation in residuals.

Range:

- 2 → no autocorrelation
- <2 → positive autocorrelation
- \>2 → negative autocorrelation

Since:

$$
0.681
$$

There is strong positive autocorrelation.

---

31. **Condition Number**

**Cond. No. = `58.4`**

Measures numerical stability and multicollinearity.

Interpretation:

- <30 → good
- 30 to 100 → moderate concern
- \>100 → serious concern

  58.4 indicates moderate numerical instability but not severe.

---

**Final Overall Interpretation**

The regression analysis shows:

- `room_num` strongly affects house price
- relationship is statistically significant
- the predictor has a strong positive coefficient
- the model explains 48.5% of variation in house prices

However:

- residuals are not normally distributed
- outliers may exist
- autocorrelation exists
- one predictor alone cannot fully explain house prices

## 2.2. Using Scikit Learn `sklearn`

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
y = df["price"]

In [ ]:
X = df[["room_num"]]

* `X` (Dependent variable) has to be a 1-D array.
* `y` (Independent variable) has to be a 2-D array.

In [ ]:
lm2 = LinearRegression()

In [ ]:
lm2.fit(X, y)

In [ ]:
print(f"Intercept: {lm2.intercept_}\nCoefficient: {lm2.coef_}")

In [ ]:
help(lm2)

In [ ]:
# Predicting the values of "y" based on the model we generated.
lm2.predict(X)

In [ ]:
sns.jointplot(x = df["room_num"], y = df["price"], data = df, kind = "reg")

# 3. Mult-Linear Regressions
## 3.1. Using `statsmodel`

In [ ]:
X_multi = df.drop("price", axis = 1)
# Dropping "price" for X variable since we need all the independent variables now and "price" is our dependent variable.
# axis = 1: When we are dropping a column.
# axis = 0: When we are dropping a row.

In [ ]:
X_multi.head()

In [ ]:
y_multi = df["price"]

In [ ]:
y_multi.head()

In [ ]:
X_multi_cons = sn.add_constant(X_multi)

In [ ]:
X_multi_cons.head()

In [ ]:
lm_multi = sn.OLS(y_multi, X_multi_cons).fit()

In [ ]:
lm_multi.summary()

**Observations:**
1. For some variables, the P-Value is nearly 0 or 0, which means that they have significant impact on dependent variable `y` (price).
2. Identify the variables with P-Values < 0.05, and also observe their coefficients for +  or - sign to make the business sense out of the data (-ve means that the price will drop the coefficient times and vice versa).

## 3.2. Using Scikit Learn `sklearn`

In [ ]:
lm3 = LinearRegression()

In [ ]:
lm3.fit(X_multi, y_multi)

In [ ]:
print(f"Intercept: {lm3.intercept_}\nCoefficient: {lm3.coef_}")

## 3.3. Test-Train Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size = 0.2, random_state = 0)
# test_size = 20 → 20% data for testing and 80% for training.

In [ ]:
print(
		f"X_train.shape: {X_train.shape}\nX_test.shape: {X_test.shape}\ny_train.shape: {y_train.shape}\ny_test.shape: {y_test.shape}"
		)

In [ ]:
lm_a = LinearRegression()

In [ ]:
lm_a.fit(X_train, y_train)

In [ ]:
y_test_a = lm_a.predict(X_test)

In [ ]:
y_train_a = lm_a.predict(X_train)

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(y_test, y_test_a)

In [ ]:
r2_score(y_train, y_train_a)

# 4. Ridge Regression
## 4.1. Standardizing the Data

In [ ]:
from sklearn import preprocessing

In [ ]:
# Scalar object to store the scaling object for X variable
scaler = preprocessing.StandardScaler().fit(X_train)

In [ ]:
X_train_s = scaler.transform(X_train)

In [ ]:
X_test_s = scaler.transform(X_test)

## 4.2. Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
lm_r = Ridge(alpha = 0.5)
# Alpha is the lambda value.

In [ ]:
lm_r.fit(X_train_s, y_train)

In [ ]:
r2_score(y_test, lm_r.predict(X_test_s))

In [ ]:
# Validation Curve to find the value of alpha for which value of r2_score is maximum
from sklearn.model_selection import validation_curve

In [ ]:
help(validation_curve)

In [ ]:
# Creating 100 sample values of alpha in the range 10^-2 to 10^8
param_range = np.logspace(-2, 8, 100)

In [ ]:
param_range

In [ ]:
train_scores, test_scores = validation_curve(
		estimator = Ridge(), X = X_train_s, y = y_train, param_name = "alpha", param_range = param_range, scoring = "r2"
		)

In [ ]:
print(f"Training Score: {train_scores}\n\nTest Score: {test_scores}")

**Observations:**
1. We did not provide any test set to `validation_curve`. It is using cross validation or K-Fold validation on training data to calculate the test scores.
2. Cross Validation `cv = 5` by default and can be set to 3.
3. For each value of lambda we are getting 5 values of r square because the validation curve is running K-Fold validation behind the scene.
4. We will take mean score of these values.

In [ ]:
train_mean = np.mean(train_scores, axis = 1)
test_mean = np.mean(test_scores, axis = 1)

In [ ]:
print(f"Training Mean: {train_mean}\n\nTest Mean: {test_mean}")

In [ ]:
# Finding the model with highest r2 score value for test.
print(f"Highest r2 score of test: {max(test_mean)}")

In [ ]:
sns.jointplot(x = np.log(param_range), y = test_mean)

In [ ]:
# Finding location (index) of the model with max mean r2 score value
np.where(test_mean == max(test_mean))

In [ ]:
# Finding out the lambda value at 31st index
param_range[31]

In [ ]:
# Fitting the Ridge Regression with this lambda value.
lm_r_best = Ridge(alpha = param_range[31])

In [ ]:
# Fitting on training dataset
lm_r_best.fit(X_train_s, y_train)

In [ ]:
r2_score(y_test, lm_r_best.predict(X_test_s))

In [ ]:
r2_score(y_train, lm_r_best.predict(X_train_s))

# 5. Lasso Regression

In [ ]:
from sklearn.linear_model import Lasso

In [ ]:
train_scores_lasso, test_scores_lasso = validation_curve(
		estimator = Lasso(), X = X_train_s, y = y_train, param_name = "alpha", param_range = param_range, scoring = "r2"
		)

In [ ]:
print(f"Training Score: {train_scores_lasso}\n\nTest Score: {test_scores_lasso}")

In [ ]:
train_mean_lasso = np.mean(train_scores_lasso, axis = 1)
test_mean_lasso = np.mean(test_scores_lasso, axis = 1)

In [ ]:
print(f"Training Mean: {train_mean_lasso}\n\nTest Mean: {test_mean_lasso}")

In [ ]:
print(f"Highest r2 score of test: {max(test_mean_lasso)}")

In [ ]:
sns.jointplot(x = np.log(param_range), y = test_mean_lasso)

In [ ]:
np.where(test_mean_lasso == max(test_mean_lasso))

In [ ]:
param_range[7]

In [ ]:
lm_l_best = Lasso(alpha = param_range[7])

In [ ]:
lm_l_best.fit(X_train_s, y_train)

In [ ]:
r2_score(y_test, lm_l_best.predict(X_test_s))

In [ ]:
r2_score(y_train, lm_l_best.predict(X_train_s))